   
# Lakeflow Expectations: Data Quality

Welcome! This demo teaches you how to enforce data quality in Lakeflow Spark Declarative Pipelines (SDP) using **expectations**.

---

## What You'll Learn

By the end of this demo, you will be able to:

1. Understand what expectations are and why they matter
2. Use `@dp.expect()` to track data quality violations
3. Use `@dp.expect_or_drop()` to filter invalid records
4. Use `@dp.expect_or_fail()` to halt pipelines on critical issues
5. Combine multiple expectations on a single table
6. Monitor data quality metrics in the pipeline UI
7. Apply expectations at different layers (bronze, silver, gold)
8. Follow data quality best practices

---

## Prerequisites

This demo builds on **Lakeflow SDP Pipeline Fundamentals**.

You should already understand:
* Streaming tables vs materialized views
* Bronze → Silver → Gold architecture
* `@dp.table` and `@dp.materialized_view` decorators
* Creating and running pipelines

**If you haven't completed the fundamentals demo, do that first!**

---

## Demo Scenario: Data Quality for Wanderbricks

We'll add data quality checks to the Wanderbricks pipeline:

**Quality checks:**
* Validate booking amounts are positive
* Ensure dates are valid
* Check ratings are in valid range (1-5)
* Verify required fields are not null
* Monitor data quality metrics

**Expectation actions:**
* **Track** violations (warnings)
* **Drop** invalid records
* **Fail** pipeline on critical issues

---

**Let's add data quality to your pipeline!**

   
## Part 1: Understanding Expectations

**Expectations** are data quality checks built into your Lakeflow SDP pipeline.

---

### **What are expectations?**

**Definition:**
* Declarative data quality rules
* SQL-like conditions that data must meet
* Applied to streaming tables and materialized views
* Monitored and tracked automatically

**Example:**
```python
from pyspark import pipelines as dp

@dp.table()
@dp.expect("valid_amount", "total_amount > 0")
def silver_bookings():
    return spark.readStream.table("bronze_bookings")
```

**What this does:**
* Checks that `total_amount > 0` for every row
* Tracks violations
* Takes action based on expectation type

---

### **Why use expectations?**

**Without expectations:**
* Bad data flows through pipeline
* Issues discovered late (in dashboards)
* Manual data quality checks
* No visibility into data quality

**With expectations:**
* Catch issues early
* Automatic data quality monitoring
* Prevent bad data from propagating
* Built-in metrics and reporting
* Self-documenting data quality rules

---

### **Where to use expectations:**

* **Bronze:** Minimal (track issues, don't drop)
* **Silver:** Extensive (drop invalid data)
* **Gold:** Critical checks (fail on issues)

---

   
### Three Types of Expectations

Lakeflow SDP has **three expectation actions** with different behaviors:

---

### **1. `@dp.expect()` - Track violations (Warning)**

**Behavior:**
* Records violation in metrics
* Keeps the invalid record
* Pipeline continues
* Tracks percentage of violations

**When to use:**
* Bronze layer (track issues, keep all data)
* Non-critical validations
* Monitoring data quality trends

**Syntax:**
```python
@dp.table()
@dp.expect("valid_amount", "total_amount > 0")
def my_table():
    ...
```

**For multiple expectations:**
```python
@dp.table()
@dp.expect_all({"valid_amount": "total_amount > 0", "valid_guests": "guests_count > 0"})
def my_table():
    ...
```

**Result:** Invalid records kept, violations tracked

---

### **2. `@dp.expect_or_drop()` - Drop invalid records**

**Behavior:**
* Drops records that violate the rule
* Records violation in metrics
* Pipeline continues
* Only valid records in output

**When to use:**
* Silver layer (clean data)
* Filter out bad data
* Ensure data quality

**Syntax:**
```python
@dp.table()
@dp.expect_or_drop("valid_amount", "total_amount > 0")
def my_table():
    ...
```

**For multiple expectations:**
```python
@dp.table()
@dp.expect_all_or_drop({"valid_amount": "total_amount > 0", "valid_guests": "guests_count > 0"})
def my_table():
    ...
```

**Result:** Invalid records dropped, violations tracked

---

### **3. `@dp.expect_or_fail()` - Fail pipeline**

**Behavior:**
* Stops pipeline execution
* Marks table as failed
* No data written
* Requires immediate attention

**When to use:**
* Gold layer (critical metrics)
* Critical business rules
* Regulatory requirements
* When bad data is unacceptable

**Syntax:**
```python
@dp.table()
@dp.expect_or_fail("valid_amount", "total_amount > 0")
def my_table():
    ...
```

**For multiple expectations:**
```python
@dp.table()
@dp.expect_all_or_fail({"valid_amount": "total_amount > 0", "valid_guests": "guests_count > 0"})
def my_table():
    ...
```

**Result:** Pipeline fails if ANY record violates

---

### **Comparison:**

| Decorator | Invalid Records | Pipeline | Use Case |
|----------|----------------|----------|----------|
| `@dp.expect()` | Kept | Continues | Track issues |
| `@dp.expect_or_drop()` | Dropped | Continues | Filter bad data |
| `@dp.expect_or_fail()` | None written | Fails | Critical rules |

---

**Tip:** Start with `@dp.expect()` to understand your data, then use `@dp.expect_or_drop()` or `@dp.expect_or_fail()`

   
## Part 2: `@dp.expect()` - Track Data Quality Issues

The `@dp.expect()` decorator tracks violations but keeps all records.

**Use case:** Monitor data quality without dropping data (especially in bronze layer)

---

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *

# ============================================
# BRONZE LAYER with @dp.expect_all()
# ============================================

@dp.table(
    name="bronze_bookings_monitored",
    comment="Raw bookings with data quality monitoring"
)
@dp.expect_all({
    "valid_amount": "total_amount > 0",
    "valid_guests": "guests_count > 0",
    "valid_dates": "check_in < check_out"
})
def bronze_bookings_monitored():
    """
    Bronze table with expectations that TRACK violations.
    Invalid records are KEPT but violations are recorded.
    """
    return (
        spark.readStream
            .table("samples.wanderbricks.bookings")
    )

   
### `@dp.expect()` - How It Works

**What we did:**

1. **Added `@dp.expect_all()` decorator** after `@dp.table`
2. **Defined 3 expectations:**
   * `valid_amount` - Amount must be positive
   * `valid_guests` - Guest count must be positive
   * `valid_dates` - Check-in before check-out

3. **Each expectation has:**
   * **Name** - Descriptive identifier ("valid_amount")
   * **Condition** - SQL expression ("total_amount > 0")

**What happens:**
* All records are kept (even invalid ones)
* Violations are tracked in metrics
* Pipeline shows warning if violations exist
* You can see violation percentage

**When to use `@dp.expect()`:**
* **Bronze layer** - Track issues, keep all data for audit
* **Monitoring** - Understand data quality trends
* **Discovery** - Learn about your data
* **Reporting** - Track quality metrics over time

**Single expectation syntax:**
```python
@dp.table()
@dp.expect("valid_amount", "total_amount > 0")
def my_table():
    ...
```

**Multiple expectations syntax:**
```python
@dp.table()
@dp.expect_all({
    "expectation_name": "SQL_condition",
    "another_check": "column IS NOT NULL",
    "range_check": "value BETWEEN 1 AND 100"
})
def my_table():
    ...
```

**Valid SQL conditions:**
* Comparison: `>`, `<`, `>=`, `<=`, `=`, `!=`
* NULL checks: `IS NOT NULL`, `IS NULL`
* Range: `BETWEEN x AND y`
* Pattern: `LIKE 'pattern'`
* Complex: `(condition1) AND (condition2)`

---

**Tip:** Use `@dp.expect()` in bronze to understand data quality before deciding on stricter rules

   
### Challenge 1: Add @dp.expect() to Bronze Reviews

**Your task:**

Add expectations to track data quality in bronze reviews.

**Requirements:**

1. Create a bronze table: `bronze_reviews_monitored`
2. Ingest from `samples.wanderbricks.reviews`
3. Add these expectations using `@dp.expect_all`:
   * `valid_rating` - Rating between 1.0 and 5.0
   * `has_rating` - Rating is not null
   * `not_deleted` - is_deleted = false

**Expected code:**
```python
from pyspark import pipelines as dp

@dp.table(
    name="bronze_reviews_monitored",
    comment="Raw reviews with quality monitoring"
)
@dp.expect_all({
    "valid_rating": "rating BETWEEN 1.0 AND 5.0",
    "has_rating": "rating IS NOT NULL",
    "not_deleted": "is_deleted = false"
})
def bronze_reviews_monitored():
    return spark.readStream.table("samples.wanderbricks.reviews")
```

**After adding:**
1. Add the code to your notebook
2. Save the notebook
3. Run the pipeline
4. Check the graph - node should show expectation metrics
5. Click the node to see violation counts

**Success criteria:** Bronze table with 3 expectations tracking violations

---

   
## Part 3: `@dp.expect_or_drop()` - Filter Invalid Data

The `@dp.expect_or_drop()` decorator removes records that violate expectations.

**Use case:** Clean data in silver layer by dropping invalid records

---

In [0]:
# ============================================
# SILVER LAYER with @dp.expect_all_or_drop()
# ============================================

@dp.table(
    name="silver_bookings_quality",
    comment="Cleaned bookings with data quality enforcement - Silver layer"
)
@dp.expect_all_or_drop({
    "valid_amount": "total_amount > 0",
    "valid_guests": "guests_count > 0 AND guests_count <= 20",
    "valid_dates": "check_in < check_out",
    "future_checkin": "check_in >= '2020-01-01'",
    "confirmed_status": "status = 'confirmed'"
})
def silver_bookings_quality():
    """
    Silver table with @dp.expect_all_or_drop.
    Records violating ANY expectation are DROPPED.
    Only clean data flows to downstream tables.
    """
    return (
        spark.readStream.table("bronze_bookings")
        .withColumn("nights", datediff(col("check_out"), col("check_in")))
        .withColumn("booking_date", to_date(col("created_at")))
        .withColumn("revenue_per_night", col("total_amount") / col("nights"))
        .select(
            "booking_id",
            "user_id",
            "property_id",
            "check_in",
            "check_out",
            "nights",
            "guests_count",
            "total_amount",
            "revenue_per_night",
            "status",
            "booking_date",
            "created_at"
        )
    )

   
### `@dp.expect_or_drop()` - How It Works

**What we did:**

1. **Added `@dp.expect_all_or_drop()` decorator** after `@dp.table`
2. **Defined 5 expectations:**
   * `valid_amount` - Positive amounts only
   * `valid_guests` - Reasonable guest count (1-20)
   * `valid_dates` - Check-in before check-out
   * `future_checkin` - No historical bookings before 2020
   * `confirmed_status` - Only confirmed bookings

**What happens:**
* Records violating ANY expectation are **dropped**
* Violations are tracked in metrics
* Only valid records in output table
* Pipeline continues (doesn't fail)

**Data flow:**
```
Bronze: 10,000 records (all data)
    ↓
    @dp.expect_all_or_drop checks
    ↓
Silver: 9,500 records (500 dropped)
```

**Benefits:**
* Automatic data cleaning
* No manual filtering needed
* Self-documenting quality rules
* Metrics show what was dropped
* Downstream tables get clean data

**When to use `@dp.expect_or_drop()`:**
* 🥈 **Silver layer** - Primary use case
* 🧹 **Data cleaning** - Remove bad records
* **Quality enforcement** - Ensure standards
* 📉 **Filtering** - Declarative filtering

**Multiple expectations:**
* Records must pass ALL expectations
* If ANY expectation fails, record is dropped
* AND logic (not OR)

---

**Tip:** `@dp.expect_or_drop()` / `@dp.expect_all_or_drop()` is the most common expectation type for silver layer

   
### Complex Expectation Conditions

Expectations can use complex SQL expressions:

**Multiple conditions (AND):**
```python
@dp.expect_or_drop("valid_guest_range", "guests_count > 0 AND guests_count <= 20")
```

**Multiple conditions (OR):**
```python
@dp.expect_or_drop("valid_status", "status IN ('confirmed', 'completed')")
```

**Date validations:**
```python
@dp.expect_all_or_drop({
    "recent_data": "created_at >= '2020-01-01'",
    "valid_date_range": "check_in BETWEEN '2020-01-01' AND '2030-12-31'"
})
```

**NULL checks:**
```python
@dp.expect_all_or_drop({
    "required_field": "column IS NOT NULL",
    "no_empty_strings": "column IS NOT NULL AND column != ''"
})
```

**Calculated conditions:**
```python
@dp.expect_all_or_drop({
    "reasonable_price": "total_amount / nights < 10000",
    "valid_percentage": "discount_pct >= 0 AND discount_pct <= 100"
})
```

**Pattern matching:**
```python
@dp.expect_all_or_drop({
    "valid_email": "email LIKE '%@%.%'",
    "valid_zip": "zip_code RLIKE '^[0-9]{5}$'"
})
```

---

**Tip:** Keep expectations simple and focused - one rule per expectation

   
### Challenge 2: Add @dp.expect_all_or_drop() to Silver Properties

**Your task:**

Create a silver properties table with data quality enforcement.

**Requirements:**

1. Create streaming table: `silver_properties_quality`
2. Read from `bronze_properties`
3. Add these `@dp.expect_all_or_drop` expectations:
   * `valid_price` - base_price > 0 AND base_price < 100000
   * `valid_guests` - max_guests > 0 AND max_guests <= 50
   * `valid_bedrooms` - bedrooms > 0
   * `has_property_type` - property_type IS NOT NULL
4. Add calculated column: `price_per_bedroom` = base_price / bedrooms
5. Select relevant columns

**Expected code:**
```python
from pyspark import pipelines as dp

@dp.table(
    name="silver_properties_quality",
    comment="Cleaned properties with quality enforcement"
)
@dp.expect_all_or_drop({
    "valid_price": "base_price > 0 AND base_price < 100000",
    "valid_guests": "max_guests > 0 AND max_guests <= 50",
    "valid_bedrooms": "bedrooms > 0",
    "has_property_type": "property_type IS NOT NULL"
})
def silver_properties_quality():
    return (
        spark.readStream.table("bronze_properties")
        .withColumn("price_per_bedroom", col("base_price") / col("bedrooms"))
        .select(
            "property_id", "host_id", "title", "property_type",
            "base_price", "price_per_bedroom", "max_guests", 
            "bedrooms", "bathrooms", "created_at"
        )
    )
```

**After adding:**
1. Add to notebook and save
2. Run pipeline
3. Check graph for the new table
4. View metrics to see how many records were dropped

**Success criteria:** Silver table dropping invalid properties

---

   
## Part 4: `@dp.expect_or_fail()` - Critical Quality Checks

The `@dp.expect_or_fail()` decorator stops the pipeline if violations occur.

**Use case:** Enforce critical business rules that cannot be violated

---

In [0]:
# ============================================
# GOLD LAYER with @dp.expect_all_or_fail()
# ============================================

@dp.materialized_view(
    name="gold_daily_revenue_critical",
    comment="Daily revenue metrics with critical quality checks - Gold layer"
)
@dp.expect_all_or_fail({
    "has_data": "booking_count > 0",
    "positive_revenue": "total_revenue > 0",
    "reasonable_avg": "avg_booking_value > 0 AND avg_booking_value < 50000"
})
def gold_daily_revenue_critical():
    """
    Gold materialized view with @dp.expect_all_or_fail.
    If ANY expectation fails, the ENTIRE PIPELINE stops.
    Use for critical business metrics.
    """
    return (
        spark.read.table("silver_bookings_quality")
        .groupBy("booking_date")
        .agg(
            count("booking_id").alias("booking_count"),
            sum("total_amount").alias("total_revenue"),
            avg("total_amount").alias("avg_booking_value"),
            sum("nights").alias("total_nights")
        )
        .orderBy("booking_date")
    )

   
### `@dp.expect_or_fail()` - How It Works

**What we did:**

1. **Added `@dp.expect_all_or_fail()` decorator** after `@dp.materialized_view`
2. **Defined 3 critical expectations:**
   * `has_data` - Must have at least 1 booking per day
   * `positive_revenue` - Revenue must be positive
   * `reasonable_avg` - Average booking value in reasonable range

**What happens:**
* If ANY expectation fails, pipeline **stops immediately**
* Table marked as **failed**
* No data written to the table
* Requires manual intervention
* Violation metrics still recorded

**When pipeline fails:**
* Graph shows red node
* Event log shows which expectation failed
* Pipeline status: "Failed"
* Downstream tables don't run
* Must fix issue and rerun

**When to use `@dp.expect_or_fail()`:**
* **Gold layer** - Critical business metrics
* **Regulatory requirements** - Must be met
* **SLA guarantees** - Data must be valid
* **Critical rules** - Cannot be violated

**Examples of critical checks:**
* Revenue must be positive
* Required fields must exist
* Data must be recent (not too old)
* Aggregations must have minimum records
* Business rules must be satisfied

**Use sparingly:**
* Too many `@dp.expect_or_fail` = frequent pipeline failures
* Only for truly critical rules
* Consider `@dp.expect_or_drop` first

---

**Tip:** Use `@dp.expect_or_fail()` only for critical rules - pipeline stops completely if violated

   
### Challenge 3: Add @dp.expect_all_or_fail() to Gold Metrics

**Your task:**

Create a gold materialized view with critical quality checks.

**Requirements:**

1. Create materialized view: `gold_property_metrics_critical`
2. Read from `silver_bookings_quality`
3. Group by `property_id`
4. Calculate:
   * Total bookings
   * Total revenue
   * Average rating (join with reviews)
5. Add `@dp.expect_all_or_fail` expectations:
   * `min_bookings` - At least 1 booking per property
   * `positive_revenue` - Revenue > 0
   * `valid_avg_rating` - Average rating between 1.0 and 5.0 (or NULL)

**Expected code:**
```python
from pyspark import pipelines as dp

@dp.materialized_view(
    name="gold_property_metrics_critical",
    comment="Property metrics with critical quality checks"
)
@dp.expect_all_or_fail({
    "min_bookings": "booking_count >= 1",
    "positive_revenue": "total_revenue > 0",
    "valid_avg_rating": "avg_rating IS NULL OR (avg_rating >= 1.0 AND avg_rating <= 5.0)"
})
def gold_property_metrics_critical():
    bookings = spark.read.table("silver_bookings_quality")
    reviews = spark.read.table("silver_reviews")
    
    return (
        bookings
        .groupBy("property_id")
        .agg(
            count("booking_id").alias("booking_count"),
            sum("total_amount").alias("total_revenue"),
            avg("nights").alias("avg_nights")
        )
        .join(
            reviews.groupBy("property_id").agg(avg("rating").alias("avg_rating")),
            "property_id",
            "left"
        )
    )
```

**After adding:**
1. Add to notebook and save
2. Run pipeline
3. Verify table appears in graph
4. Check if pipeline succeeds (all expectations pass)
5. Try modifying an expectation to make it fail (for testing)

**Success criteria:** Gold materialized view with critical expectations

---

   
## Part 5: Combining Multiple Expectation Types 🔗

You can stack multiple expectation decorators on the same table!

**Why combine?**
* Different severity levels for different rules
* Track some issues, drop others, fail on critical
* Flexible data quality strategy

---

In [0]:
# ============================================
# COMBINING MULTIPLE EXPECTATION TYPES
# ============================================

@dp.table(
    name="silver_bookings_comprehensive",
    comment="Bookings with comprehensive data quality checks"
)
# Track these issues (keep records)
@dp.expect_all({
    "recent_booking": "created_at >= '2023-01-01'",
    "reasonable_nights": "nights <= 365"
})
# Drop records with these issues
@dp.expect_all_or_drop({
    "valid_amount": "total_amount > 0",
    "valid_guests": "guests_count > 0 AND guests_count <= 20",
    "valid_dates": "check_in < check_out",
    "confirmed_only": "status = 'confirmed'"
})
def silver_bookings_comprehensive():
    """
    Silver table with multiple expectation types:
    - @dp.expect_all: Track old bookings and long stays (warnings)
    - @dp.expect_all_or_drop: Remove invalid data (filtering)
    """
    return (
        spark.readStream.table("bronze_bookings")
        .withColumn("nights", datediff(col("check_out"), col("check_in")))
        .withColumn("booking_date", to_date(col("created_at")))
        .select(
            "booking_id",
            "user_id",
            "property_id",
            "check_in",
            "check_out",
            "nights",
            "guests_count",
            "total_amount",
            "status",
            "booking_date",
            "created_at"
        )
    )

   
### Combining Expectations - How It Works

**What we did:**

1. **Stacked both `@dp.expect_all()` and `@dp.expect_all_or_drop()` decorators** on the same table
2. **Different actions for different rules:**
   * `@dp.expect_all` - Track old bookings (warning, keep data)
   * `@dp.expect_all_or_drop` - Remove invalid data (filtering)

**Execution order:**
```
1. Read data from source
2. Apply transformations
3. Evaluate ALL expectations
4. expect: Track violations, keep records
5. expect_or_drop: Drop violating records
6. Write valid records to table
```

**Key points:**
* You can stack multiple expectation decorators on one table
* Each decorator type acts independently
* `@dp.expect_all` tracks but keeps records
* `@dp.expect_all_or_drop` removes violating records
* All violations are tracked in metrics

**Combining all three types:**
```python
@dp.table()
@dp.expect_all({"track_this": "condition"})            # Track (keep records)
@dp.expect_all_or_drop({"filter_this": "condition"})    # Drop invalid
@dp.expect_all_or_fail({"critical_rule": "condition"})  # Fail on violation
def my_table():
    ...
```

**When to combine:**
* Different severity for different rules
* Track some issues, drop others
* Fail on critical but filter non-critical

---

**Tip:** Use `@dp.expect_all` for monitoring and `@dp.expect_all_or_drop` for filtering on the same table

### Challenge 4: Comprehensive Quality Checks

**Your task:**

Create a silver table with all three expectation types.

**Requirements:**

1. Create streaming table: `silver_reviews_comprehensive`
2. Read from `bronze_reviews`
3. Add **all three** expectation types:

**@dp.expect_all (track):**
* `has_comment` - comment IS NOT NULL (track reviews without comments)
* `long_comment` - LENGTH(comment) > 10 (track short comments)

**@dp.expect_all_or_drop (filter):**
* `valid_rating` - rating BETWEEN 1.0 AND 5.0
* `not_deleted` - is_deleted = false
* `has_property` - property_id IS NOT NULL

**@dp.expect_or_fail (critical):**
* `has_data` - review_id IS NOT NULL (critical - must have ID)

4. Add calculated columns:
   * `review_date` = DATE(created_at)
   * `comment_length` = LENGTH(comment)

**Expected code:**
```python
from pyspark import pipelines as dp

@dp.table(
    name="silver_reviews_comprehensive",
    comment="Reviews with comprehensive quality checks"
)
@dp.expect_all({
    "has_comment": "comment IS NOT NULL",
    "long_comment": "LENGTH(comment) > 10"
})
@dp.expect_all_or_drop({
    "valid_rating": "rating BETWEEN 1.0 AND 5.0",
    "not_deleted": "is_deleted = false",
    "has_property": "property_id IS NOT NULL"
})
@dp.expect_or_fail("has_data", "review_id IS NOT NULL")
def silver_reviews_comprehensive():
    return (
        spark.readStream.table("bronze_reviews")
        .withColumn("review_date", to_date(col("created_at")))
        .withColumn("comment_length", length(col("comment")))
        .select(
            "review_id", "booking_id", "property_id", "user_id",
            "rating", "comment", "comment_length", "review_date", "created_at"
        )
    )
```

**After adding:**
1. Add to notebook and save
2. Run pipeline
3. View data quality metrics
4. Check how many records were dropped
5. Check warning counts

**Success criteria:** Table with all 3 expectation types working

---

## Part 6: Monitoring Data Quality Metrics

Lakeflow SDP automatically tracks and displays data quality metrics.

---

### View Expectations in Pipeline Graph

**Follow these steps:**

1. Run your pipeline with expectations
2. Go to the **"Graph"** tab
3. Look at nodes with expectations

**What you'll see:**

**Nodes with expectations show:**
* **Green checkmark** - All expectations passed
* **Yellow warning** - Some @dp.expect() violations (tracked)
* **Red X** - @dp.expect_or_fail violations (failed)
* **Metrics badge** - Number of expectations

**Click on a node:**
* Side panel opens
* Shows **"Data Quality"** section
* Lists all expectations
* Shows pass/fail status
* Shows violation counts

---

### Query Expectation Metrics

**Lakeflow SDP stores expectation metrics** that you can query!

**System tables:**

Lakeflow SDP creates system tables with metrics:
* `system.lakeflow.event_log` - Pipeline execution events
* Table-specific metrics in pipeline metadata

**Query expectation metrics:**
```sql
-- View data quality metrics from event log
SELECT 
  timestamp,
  details:flow_definition.output_dataset AS table_name,
  details:flow_definition.expectations AS expectations,
  details:flow_progress.metrics.num_output_rows AS output_rows
FROM system.lakeflow.event_log
WHERE event_type = 'flow_progress'
ORDER BY timestamp DESC
```

---

**Tip:** Use system tables to build data quality dashboards and track trends over time

## Troubleshooting Expectations

**Expectation always failing:**
* Check SQL condition syntax
* Verify column names are correct
* Test condition in regular SQL query first
* Check if condition is too strict
* Review sample data

**High violation rates:**
* Review expectation condition
* Check if upstream data changed
* Analyze violated records
* Adjust threshold if needed
* Fix upstream data quality

**Pipeline failing with @dp.expect_or_fail:**
* Check event log for details
* Review which expectation failed
* Query source data to understand issue
* Consider changing to @dp.expect_or_drop temporarily
* Fix root cause

**Expectations not showing in UI:**
* Verify expectation syntax is correct
* Check decorator placement (after @dp.table, before def)
* Ensure pipeline ran successfully
* Refresh the page

**Performance issues:**
* Reduce number of expectations
* Simplify complex conditions
* Use indexed columns
* Profile slow expectations

---

**Tip:** Test expectations on sample data before applying to full pipeline

## Congratulations!

You've completed the Lakeflow SDP Expectations demo!

### **What You Learned:**

* **Expectations concept** - Declarative data quality rules  
* **@dp.expect()** - Track violations, keep records  
* **@dp.expect_or_drop()** - Filter invalid data  
* **@dp.expect_or_fail()** - Halt on critical issues  
* **Combining expectations** - Stack multiple decorator types on one table  
* **Monitoring metrics** - View quality in UI  
* **Best practices** - When and how to use each type  
* **Common patterns** - Reusable expectation templates  

---

### **Key Takeaways:**

1. **Expectations are declarative** - Define rules, Lakeflow SDP enforces
2. **Three types for three actions** - Track, drop, or fail
3. **Bronze tracks, silver drops, gold fails** - General pattern
4. **Metrics are automatic** - No manual tracking needed
5. **Combine types for flexibility** - Different rules, different actions
6. **Start lenient, get stricter** - @dp.expect → @dp.expect_or_drop → @dp.expect_or_fail
7. **Monitor trends** - Increasing violations = upstream issues

---

### **Next Steps:**

* Add expectations to your existing pipelines
* Build data quality dashboards
* Set up alerts for quality degradation
* Create expectation libraries
* Document data quality standards
* Train team on expectation patterns
* Explore the Lakeflow Pipelines Editor for multi-file development

---

### **Resources:**

* [Lakeflow SDP Expectations](https://docs.databricks.com/aws/en/ldp/expectations)
* [Expectation Patterns](https://docs.databricks.com/aws/en/ldp/expectation-patterns)
* [SDP Best Practices](https://docs.databricks.com/aws/en/ldp/best-practices)
* [SDP Python Reference](https://docs.databricks.com/aws/en/ldp/developer/ldp-python-ref-expectations)

